In [ ]:
# ===================================================================================
# VECTOR TILE LAYER MIGRATION SCRIPT
# Migrates Vector Tile Layers from Source (10.9.1) to Target (11.3).
# - Exports as Vector Tile Package (.vtpk), downloads, uploads to target, publishes.
# - Falls back to direct download if export fails.
# - Copies metadata, thumbnails, sharing, and owner/folder assignments.
# ===================================================================================

import pandas as pd
import json
import copy
import time
import csv
import os
import datetime
import urllib3
import requests
import sys
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from arcgis.gis import GIS

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# =============================================================================
# --- CONFIGURATION (from shared config) ---------------------------------------
# =============================================================================
from migration_config import *

# --- ORCHESTRATOR SIDECAR LOADER ---
_sidecar_path = os.path.join(os.path.dirname(os.path.abspath("__file__")), "_sidecar_9_vector_tiles.json")
if os.path.exists(_sidecar_path):
    import json as _json
    VECTOR_TILE_IDS = _json.load(open(_sidecar_path))["ids"]
    print(f"[Orchestrator] Loaded {len(VECTOR_TILE_IDS)} Vector Tile Layer IDs from sidecar.")
else:
    VECTOR_TILE_IDS = [
        # Example Source IDs
        # "04950701a5744339acb47bbc43602295",
    ]

# =============================================================================
# --- LOGGING & CONNECT --------------------------------------------------------
# =============================================================================
if not os.path.exists(TEMP_DIR):
    os.makedirs(TEMP_DIR)

STATS = {
    "scanned": 0,
    "created": 0,
    "skipped_log": 0,
    "failures": 0,
}

ALREADY_MIGRATED_IDS = set()
if os.path.exists(LOG_FILE):
    try:
        df_log = pd.read_csv(LOG_FILE)
        if "SourceID" in df_log.columns:
            ALREADY_MIGRATED_IDS = set(df_log["SourceID"].astype(str).str.strip())
        print(f"Loaded history: {len(ALREADY_MIGRATED_IDS)} items previously migrated.")
    except:
        pass
else:
    try:
        with open(LOG_FILE, mode="w", newline="") as f:
            csv.writer(f).writerow(["SourceID", "LayerIndex", "TargetID", "Title", "MigratedDate", "Type"])
    except:
        pass


def log_migration(source_id, target_id, title):
    try:
        with open(LOG_FILE, mode="a", newline="") as f:
            csv.writer(f).writerow([
                source_id, "", target_id, title,
                datetime.datetime.now().strftime("%m/%d/%Y %H:%M"), "Vector Tile Layer"
            ])
    except:
        print("   ⚠ Log Write Failed")


print("Connecting...")
try:
    session = requests.Session()
    retry_strategy = Retry(
        total=5,
        backoff_factor=2,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["HEAD", "GET", "POST"]
    )
    adapter = HTTPAdapter(max_retries=retry_strategy)
    session.mount("https://", adapter)
    session.mount("http://", adapter)

    source_gis = GIS(url=SOURCE_URL, token=SOURCE_TOKEN, verify_cert=False, referer=SOURCE_URL, session=session)
    target_gis = GIS(url=TARGET_URL, token=TARGET_TOKEN, verify_cert=False, referer=TARGET_URL, session=session)

    if not target_gis.users.me:
        raise Exception("Target login failed.")
    print(f" > Source: {source_gis.users.me.username}")
    print(f" > Target: {target_gis.users.me.username}")

except Exception as e:
    print(f"❌ CONNECTION FAILURE: {e}")
    raise SystemExit(1)

# =============================================================================
# --- HELPER: FOLDER & OWNER UTILS ---------------------------------------------
# =============================================================================
def get_source_folder_name(source_item):
    """Retrieve the folder name of the source item."""
    try:
        if not source_item.ownerFolder:
            return None
        user = source_gis.users.get(source_item.owner)
        if user:
            for f in user.folders:
                fid = f['id'] if isinstance(f, dict) else f.id
                if fid == source_item.ownerFolder:
                    return f['title'] if isinstance(f, dict) else f.title
    except:
        pass
    return None


def ensure_target_folder_exists(username, folder_name):
    """Ensure a folder exists for a specific user in Target."""
    try:
        user = target_gis.users.get(username)
        if not user:
            return False

        existing_folders = [f['title'] if isinstance(f, dict) else f.title for f in user.folders]
        if folder_name in existing_folders:
            return True

        print(f"      + Creating folder '{folder_name}' for user '{username}'...")
        target_gis.content.create_folder(folder_name, owner=username)
        return True
    except Exception as e:
        print(f"      ⚠ Folder creation error: {e}")
        return False


def assign_correct_owner_and_folder(source_item, target_item):
    """Reassign the target item to the correct owner and folder based on source."""
    try:
        source_owner = source_item.owner
        target_owner_to_use = DEFAULT_OWNER
        target_folder_to_use = DEFAULT_FOLDER

        found_user = target_gis.users.get(source_owner)

        if found_user:
            print(f"      👤 Source owner '{source_owner}' found in target.")
            target_owner_to_use = source_owner

            src_folder_name = get_source_folder_name(source_item)
            if src_folder_name:
                if ensure_target_folder_exists(target_owner_to_use, src_folder_name):
                    target_folder_to_use = src_folder_name
                else:
                    print(f"      ⚠ Could not create folder '{src_folder_name}'. Using Default.")
            else:
                target_folder_to_use = None  # Root folder
        else:
            print(f"      👤 Source owner '{source_owner}' NOT found. Defaulting to '{DEFAULT_OWNER}'.")
            ensure_target_folder_exists(DEFAULT_OWNER, DEFAULT_FOLDER)

        print(f"      📂 Moving to: Owner={target_owner_to_use}, Folder={target_folder_to_use}")
        target_item.reassign_to(target_owner_to_use, target_folder=target_folder_to_use)

    except Exception as owner_err:
        print(f"      ⚠ Ownership assignment failed: {owner_err}")


# =============================================================================
# --- HELPER: COPY THUMBNAIL ---------------------------------------------------
# =============================================================================
def copy_thumbnail(source_item, target_item):
    """Download thumbnail from source item and upload to target item."""
    try:
        print("      🖼️ Copying Thumbnail...")
        temp_dir = os.path.join(TEMP_DIR, "thumbnails_temp")
        os.makedirs(temp_dir, exist_ok=True)
        thumb_path = source_item.download_thumbnail(save_folder=temp_dir)
        if thumb_path:
            target_item.update(thumbnail=thumb_path)
            print("         ✅ Thumbnail copied.")
        else:
            print("         - No thumbnail found on source item.")
    except Exception as e:
        print(f"      ⚠ Thumbnail copy failed: {e}")


# =============================================================================
# --- HELPER: MIRROR SHARING (STRICT SOURCE MATCH) -----------------------------
# =============================================================================
def mirror_source_sharing(source_item, target_item):
    """
    1. Checks Source Sharing (Public/Org/Private).
    2. STRICTLY applies the same level to Target.
    3. Maps Source Groups -> Target Groups (by exact Title).
    """
    try:
        print("      👥 Mirroring Sharing Permissions (Source -> Target)...")

        # 1. Access Level
        is_public = False
        is_org = False

        if source_item.access == 'public':
            is_public = True
            is_org = True
        elif source_item.access == 'org':
            is_org = True

        # 2. Get List of Groups (Robust Dictionary Check)
        source_group_list = []
        raw_shared = source_item.shared_with

        if isinstance(raw_shared, dict) and 'groups' in raw_shared:
            source_group_list = raw_shared['groups']
        elif isinstance(raw_shared, list):
            source_group_list = raw_shared

        target_group_ids = []

        if source_group_list:
            print(f"         - Found {len(source_group_list)} source groups. Mapping...")
            for sg in source_group_list:
                sg_title = sg.title if hasattr(sg, 'title') else str(sg)

                found_groups = target_gis.groups.search(f"title:\"{sg_title}\"")
                matched_group = next((g for g in found_groups if g.title == sg_title), None)

                if matched_group:
                    target_group_ids.append(matched_group.id)
                    print(f"           + Mapped: '{sg_title}'")
                else:
                    print(f"           - Skipped: '{sg_title}' (Not found in Target)")

        # 3. Apply
        target_item.share(everyone=is_public, org=is_org, groups=target_group_ids)
        print(f"         ✅ Shared (Org={is_org}, Public={is_public}, Groups={len(target_group_ids)})")

    except Exception as e:
        print(f"      ⚠ Sharing Mirror Failed: {e}")


# =============================================================================
# --- MAIN LOOP ----------------------------------------------------------------
# =============================================================================
print("\nStarting Vector Tile Layer Migration...")
start_time = time.time()

for source_id in VECTOR_TILE_IDS:
    try:
        STATS["scanned"] += 1

        # CHECK 1: HISTORY LOG
        if str(source_id) in ALREADY_MIGRATED_IDS:
            print(f"\n[Skip] {source_id} found in migration log.")
            STATS["skipped_log"] += 1
            continue

        # GET SOURCE ITEM
        source_item = source_gis.content.get(source_id)
        if not source_item:
            print(f"❌ Source item {source_id} not found.")
            STATS["failures"] += 1
            continue

        print(f"\nProcessing: {source_item.title} ({source_id})...")

        # DETERMINE TARGET TITLE
        target_title = source_item.title
        if APPEND_MIGRATED:
            target_title = f"{source_item.title} (Migrated)"

        export_name = f"Migrate_{source_id}_{int(time.time())}"
        local_path = None
        exported_item = None

        # --- PLAN A: EXPORT AS VECTOR TILE PACKAGE ---
        print(" > 1. Attempting Vector Tile Package Export...")
        try:
            exported_item = source_item.export(title=export_name, export_format="Vector Tile Package")
            local_path = exported_item.download(save_path=TEMP_DIR)
            print("   ✅ Export successful.")
        except Exception as export_err:
            print(f"   ⚠ Export failed: {export_err}")
            print("   Attempting direct download as fallback...")
            # --- PLAN B: DIRECT DOWNLOAD ---
            try:
                local_path = source_item.download(save_path=TEMP_DIR)
                if local_path:
                    print(f"   ✅ Direct download successful: {local_path}")
                else:
                    raise Exception("Download returned None")
            except Exception as dl_err:
                print(f"   ❌ Direct download also failed: {dl_err}")
                STATS["failures"] += 1
                continue
        finally:
            if exported_item:
                try:
                    exported_item.delete()
                except:
                    pass

        if not local_path or not os.path.exists(local_path):
            print(f"   ❌ Local file missing: {local_path}")
            STATS["failures"] += 1
            continue

        # --- UPLOAD TO TARGET ---
        print(" > 2. Uploading to Target...")
        item_tags = list(source_item.tags) if source_item.tags else []
        if "Migrated" not in item_tags:
            item_tags.append("Migrated")
        item_tags.append(f"SourceID:{source_id}")

        item_properties = {
            "title": target_title,
            "type": "Vector Tile Package",
            "tags": ", ".join(item_tags),
            "snippet": source_item.snippet or "",
            "description": source_item.description or "",
            "accessInformation": source_item.accessInformation or "",
            "licenseInfo": source_item.licenseInfo or "",
        }

        uploaded_item = target_gis.content.add(item_properties=item_properties, data=local_path)
        if not uploaded_item:
            print("   ❌ Upload failed.")
            STATS["failures"] += 1
            continue

        # --- PUBLISH ---
        print(" > 3. Publishing Vector Tile Layer...")
        try:
            new_item = uploaded_item.publish()
        except Exception as pub_err:
            print(f"   ❌ Publish failed: {pub_err}")
            try:
                uploaded_item.delete()
            except:
                pass
            STATS["failures"] += 1
            continue

        if not new_item:
            print("   ❌ Publish returned None.")
            STATS["failures"] += 1
            continue

        print(f"   ✅ Published: {new_item.id} - {new_item.title}")
        STATS["created"] += 1
        ALREADY_MIGRATED_IDS.add(str(source_id))

        # --- COPY METADATA ---
        print(" > 4. Updating Metadata...")
        try:
            new_item.update(item_properties={
                "tags": ", ".join(item_tags),
                "snippet": source_item.snippet or "",
                "description": source_item.description or "",
                "accessInformation": source_item.accessInformation or "",
                "licenseInfo": source_item.licenseInfo or "",
            })
        except Exception as meta_err:
            print(f"      ⚠ Metadata update warning: {meta_err}")

        # --- COPY THUMBNAIL ---
        copy_thumbnail(source_item, new_item)

        # --- MIRROR SHARING (BEFORE OWNER REASSIGNMENT) ---
        mirror_source_sharing(source_item, new_item)

        # --- ASSIGN OWNER & FOLDER ---
        print(" > 5. Assigning Owner & Folder...")
        assign_correct_owner_and_folder(source_item, new_item)

        # --- LOG MIGRATION ---
        log_migration(source_id, new_item.id, new_item.title)

        # --- CLEAN UP TEMP FILES ---
        try:
            if local_path and os.path.exists(local_path):
                os.remove(local_path)
            # Also try to clean up the uploaded package item
            try:
                uploaded_item.delete()
            except:
                pass
        except:
            pass

        print(f"   💤 Cooling down for {THROTTLE_SECONDS} seconds...")
        time.sleep(THROTTLE_SECONDS)

    except Exception as e:
        print(f"❌ FAILED on {source_id}: {e}")
        STATS["failures"] += 1

# =============================================================================
# --- REPORT -------------------------------------------------------------------
# =============================================================================
end_time = time.time()
total_seconds = int(end_time - start_time)
duration_str = f"{total_seconds // 3600}h {(total_seconds % 3600) // 60}m {total_seconds % 60}s"

print("\n" + "=" * 60)
print("      VECTOR TILE LAYER MIGRATION REPORT")
print("=" * 60)
print(f" ⏱️  Total Duration:             {duration_str}")
print("-" * 60)
print(f" 📦 Items Scanned:             {STATS['scanned']}")
print(f" 🧾 Skipped (log):             {STATS['skipped_log']}")
print(f" 🚀 Vector Tiles Created:      {STATS['created']}")
print(f" ❌ Failures:                  {STATS['failures']}")
print("=" * 60)